In [2]:
# !pip install pymorphy3

In [4]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/papa/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Выполните лемматизацию на целом предложении text и сравните со стеммингом на предмет сохранения корректности форм слова и контекста.

Выполните задание локально, а затем проведите самопроверку, изучив авторское решение. 

In [6]:
from pymorphy3 import MorphAnalyzer

from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize


text = "Дети играли в снежки и пришли домой мокрые, но довольные"

# Инициализируем стеммер и выполняем токенизацию
stemmer = SnowballStemmer("russian")
tokens = word_tokenize(text, language="russian") 

stems = [stemmer.stem(token) for token in tokens]
print("Стемминг (nltk):", stems)

# Создаём список из лемм
lemms_pm = [MorphAnalyzer().parse(token)[0].normal_form for token in tokens]
print("Леммы (pymorphy)", lemms_pm)

Стемминг (nltk): ['дет', 'игра', 'в', 'снежк', 'и', 'пришл', 'дом', 'мокр', ',', 'но', 'довольн']
Леммы (pymorphy) ['ребёнок', 'играть', 'в', 'снежок', 'и', 'прислать', 'домой', 'мокрый', ',', 'но', 'довольный']


Вам предстоит выполнить полную предобработку [корпуса медиастатей](https://code.s3.yandex.net/deep-learning/media_articles.csv) на русском языке, посвящённых разным видам спорта, а затем, используя матрицу TF-IDF, классифицировать документы с помощью логистической регрессии. Напишите свою реализацию функции, выполняющей предобработку текста `tokenize_lemmatize`: токенизация + лемматизация + приведение к нижнему регистру + удаление всех чисел из текста.

In [7]:
import os
import requests

url = 'https://code.s3.yandex.net/deep-learning/media_articles.csv'
filename = 'data/media_articles.csv'
os.makedirs(os.path.dirname(filename), exist_ok=True)

response = requests.get(url)

if response.status_code == 200:
    with open(filename, 'w', encoding=response.encoding) as file:
        file.write(response.text)
    print(f"Файл успешно сохранён как {filename}")
else:
    print(f"Ошибка загрузки: статус {response.status_code}")

Файл успешно сохранён как data/media_articles.csv


In [2]:
import pymorphy3
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords


[nltk_data] Downloading package punkt to /Users/papa/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/papa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
df = pd.read_csv("data/media_articles.csv")
train, test = train_test_split(df, test_size=0.25, stratify=df["category"], random_state=42)
  
# загружаем стоп-слова
stop_words = stopwords.words("russian")

# инициализируем лемматизатор
morph = pymorphy3.MorphAnalyzer()

# функция предобработки текстов
def tokenize_lemmatize(text):
    """убираем стоп-слова и числа, приводим к нижнему регистру, лемматизируем"""
    tokens = word_tokenize(text.lower(), language="russian")
    tokens = [token for token in tokens if token not in stop_words and not token.isdigit()]
    tokens = [morph.parse(token)[0].normal_form for token in tokens]
    return tokens


# инициализируем и применяем для train датасета TF-IDF
tf_idf = TfidfVectorizer(tokenizer=tokenize_lemmatize,
                         min_df=2,
                         max_df=0.95,
                         max_features=10_000)
tf_idf_matrix = tf_idf.fit_transform(train["text"], )

# решаем задачу классификации с помощью логистической регрессии
clf = LogisticRegression(C=0.1, random_state=42)
clf.fit(tf_idf_matrix, train["category"])

# визуализируем качество классификации
print(classification_report(test["category"], clf.predict(tf_idf.transform(test["text"]))))

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 4/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

   athletics       0.91      0.75      0.82       196
   autosport       0.74      0.83      0.78       227
  basketball       0.94      0.55      0.70       184
     extreme       0.50      0.84      0.63       208
    football       0.81      0.70      0.75       209
      hockey       0.74      0.83      0.78       218
   motosport       0.88      0.85      0.86       204
      tennis       0.90      0.94      0.92       220
  volleyball       0.85      0.67      0.75       208
winter_sport       0.81      0.77      0.79       205

    accuracy                           0.78      2079
   macro avg       0.81      0.77      0.78      2079
weighted avg       0.81      0.78      0.78      2079

